In [ ]:
import requests
import pandas as pd
from datetime import datetime
import pytz  # <-- NOVO IMPORT

# =========================
# FUNÇÃO DE CONVERSÃO DE DATA (ADICIONAR AQUI)
# =========================
def converter_data_trello(data_str):
    """Converte data do Trello (UTC) para horário de Brasília (UTC-3)"""
    if not data_str or pd.isna(data_str):
        return pd.NaT
    dt_utc = pd.to_datetime(data_str, utc=True, errors='coerce')
    if pd.isna(dt_utc):
        return pd.NaT
    return dt_utc.tz_convert('America/Sao_Paulo')

# =========================
# CONFIGURAÇÕES
# =========================
NOME_QUADRO = 

AUTH = {
    
    
}

# =========================
# FUNÇÃO DE REQUISIÇÃO
# =========================
def trello_get(endpoint, params=None):
    url = f"https://api.trello.com/1/{endpoint}"
    response = requests.get(url, params={**AUTH, **(params or {})})
    response.raise_for_status()
    return response.json()

# =========================
# 1. BUSCAR QUADRO
# =========================
boards = trello_get("members/me/boards")
board = next((b for b in boards if b['name'] == NOME_QUADRO), None)
if not board:
    raise Exception(f"Quadro '{NOME_QUADRO}' não encontrado")

print(f"✅ Conectado: {board['name']} (ID: {board['id']})")

# =========================
# 2. BUSCAR CARDS (UMA REQUISIÇÃO)
# =========================
cards = trello_get(f"boards/{board['id']}/cards")
print(f"✅ {len(cards)} cards encontrados")

# =========================
# 3. BUSCAR TODAS AS AÇÕES DO BOARD (UMA REQUISIÇÃO, NÃO UMA POR CARD!)
# =========================
# Filtro: 'updateCard' já reduz volume de dados
all_actions = trello_get(
    f"boards/{board['id']}/actions",
    params={"filter": "updateCard"}
)
print(f"✅ {len(all_actions)} ações do tipo updateCard encontradas")

# =========================
# 4. PROCESSAR DADOS (SEM LOOP ANINHADO DESNECESSÁRIO)
# =========================
cards_dict = {card['id']: card for card in cards}

acoes_list = []
lead_times_list = []

for action in all_actions:
    card_id = action['data']['card']['id']
    card = cards_dict.get(card_id, {})
    
    # Ações (movimentações)
    acoes_list.append({
        'card_id': card_id,
        'card_name': card.get('name'),
        'action_date': converter_data_trello(action['date']),
        'list_before': action['data'].get('listBefore', {}).get('name'),
        'list_after': action['data'].get('listAfter', {}).get('name'),
        'action_type': action['type']
    })

# Processar lead time por card
for card in cards:
    card_actions = [a for a in all_actions if a['data']['card']['id'] == card['id']]
    
    # Data de criação (do próprio card)
    data_criacao = converter_data_trello(card.get('dateLastActivity'))
    
    # Última movimentação para "Concluido"
    mov_concluido = [
        a for a in card_actions
        if a['data'].get('listAfter', {}).get('name') == 'Concluido'
    ]
    
    if mov_concluido:
        ultima_conclusao = mov_concluido[-1]
        data_conclusao = converter_data_trello(ultima_conclusao['date'])
        lead_time_dias = (data_conclusao - data_criacao).days if data_criacao and data_conclusao else None
        status = 'Concluído'
    else:
        data_conclusao = None
        lead_time_dias = None
        status = 'Não concluído'
    
    lead_times_list.append({
        'card_id': card['id'],
        'card_name': card['name'],
        'data_criacao': data_criacao,
        'data_ultima_atividade': converter_data_trello(card.get('dateLastActivity')),
        'data_conclusao': data_conclusao,
        'lead_time_dias': lead_time_dias,
        'status': status
    })

# =========================
# 5. DATAFRAMES (DATAS COMO DATETIME)
# =========================
df_acoes = pd.DataFrame(acoes_list)
df_lead = pd.DataFrame(lead_times_list)
df_cards = pd.DataFrame(cards)

# =========================
# 6. EXIBIR NO CONSOLE (FORMATO LEGÍVEL)
# =========================
print("\n📊 AMOSTRA LEAD TIME (datas em formato legível):")
display_df = df_lead.copy()
for col in ['data_criacao', 'data_ultima_atividade', 'data_conclusao']:
    if col in display_df.columns:
        display_df[col] = pd.to_datetime(display_df[col]).dt.strftime('%d/%m/%Y %H:%M:%S')
print(display_df[['card_name', 'data_criacao', 'lead_time_dias', 'status']].head(10).to_string())

print(f"\n✅ {len(df_acoes)} ações registradas")
print(f"✅ DataFrames prontos com datas em datetime para uso futuro")

# =========================
# 7. (OPCIONAL) DESCOMENTE ABAIXO PARA EXPORTAR CSV
# =========================
# df_cards.to_csv('cards_trello.csv', index=False, sep=';')
# df_acoes.to_csv('acoes_trello.csv', index=False, sep=';')
# df_lead.to_csv('lead_time.csv', index=False, sep=';')
# print("\n✅ Arquivos CSV gerados")

✅ Conectado: CTI  (ID: 69fd20465fb2b7d1fb85e050)
✅ 20 cards encontrados
✅ 50 ações do tipo updateCard encontradas

📊 AMOSTRA LEAD TIME (datas em formato legível):
                card_name         data_criacao  lead_time_dias         status
0                 ticket:  11/05/2026 19:14:52             NaN  Não concluído
1      Consultor: Janaina  11/05/2026 19:14:51             NaN  Não concluído
2   Documento de abertura  11/05/2026 18:59:16             NaN  Não concluído
3  Analista: Tiago Madrid  11/05/2026 19:04:11            -1.0      Concluído
4  Documento de conclusão  11/05/2026 19:04:12             NaN  Não concluído
5             Alinhamento  08/05/2026 20:12:26             NaN  Não concluído
6                Kick-Off  08/05/2026 20:12:54             NaN  Não concluído
7            Configuração  08/05/2026 20:13:10             NaN  Não concluído
8            Double Check  08/05/2026 20:16:22             NaN  Não concluído
9  Encaminhar dispositivo  08/05/2026 20:16:38           